In [1]:
# ========================= #
# Imports and configuration #
# ========================= #

import os
from datasets import load_dataset
from google.cloud import storage
from dotenv import load_dotenv
import pandas as pd

# Import seed and utilities
import sys
sys.path.append('../../src')
from utils import set_seed, SEED

set_seed()

# GCP configuration
PROJECT_ID = "project-5c89dcac-34cb-453d-bd7"
BUCKET_RAW = "sinodio-raw-data"

# HuggingFace authentication
load_dotenv()
from huggingface_hub import login
login(token=os.getenv("HUGGINGFACE_TOKEN"))

# Datasets to download
DATASETS = {
    "spanish_hate_superset": ("manueltonneau/spanish-hate-speech-superset", None),
    "detoxis": ("iberbench/iberbench_all", 
                "iberlef-detoxis-toxicity_detection-2021-spanish"),
    "haha": ("iberbench/iberbench_all", 
             "iberlef-haha-humor_detection-2021-spanish"),
}

In [2]:
# ===================== #
# Initialize GCP client #
# ===================== #

# Authenticate using Application Default Credentials
os.environ["GCLOUD_PROJECT"] = PROJECT_ID
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_RAW)

print(f"Connected to bucket: {BUCKET_RAW}")

Connected to bucket: sinodio-raw-data


In [3]:
# ============================ #
# Download_and_upload function #
# ============================ #

def download_and_upload(dataset_name, config_name, destination_folder):
    """
    Downloads a dataset from HuggingFace and uploads it 
    to GCP Cloud Storage as CSV files.
    """
    print(f"Downloading {dataset_name}...")
    
    if config_name:
        dataset = load_dataset(dataset_name, config_name)
    else:
        dataset = load_dataset(dataset_name)
    
    for split in dataset.keys():
        # Convert to pandas dataframe
        df = dataset[split].to_pandas()
        
        # Save locally as temporary file
        temp_path = f"/tmp/{destination_folder}_{split}.csv"
        df.to_csv(temp_path, index=False)
        
        # Upload to GCP bucket
        blob_path = f"{destination_folder}/{split}.csv"
        blob = bucket.blob(blob_path)
        blob.upload_from_filename(temp_path)
        
        print(f"Uploaded {blob_path} ({len(df)} rows)")
        
        # Clean up temporary file
        os.remove(temp_path)

In [4]:
# ================ #
# Run the pipeline #
# ================ #

for dataset_key, (dataset_name, config_name) in DATASETS.items():
    download_and_upload(dataset_name, config_name, dataset_key)

print("All datasets uploaded successfully.")

es_hf_102024.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29855 [00:00<?, ? examples/s]

Uploaded spanish_hate_superset/train.csv (29855 rows)


README.md: 0.00B [00:00, ?B/s]

iberlef-detoxis-toxicity_detection-2021-(…):   0%|          | 0.00/486k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3463 [00:00<?, ? examples/s]

Uploaded detoxis/train.csv (3463 rows)


iberlef-haha-humor_detection-2021-spanis(…):   0%|          | 0.00/1.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24000 [00:00<?, ? examples/s]

Uploaded haha/train.csv (24000 rows)
All datasets uploaded successfully.
